In [1]:
FILE_PATH = "../data/Credit Risk Benchmark Dataset.csv"

In [2]:
import pandas as pd
import numpy as np
import scipy.stats as ss
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import SimpleImputer, KNNImputer, IterativeImputer
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.linear_model import BayesianRidge, LogisticRegression
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, mean_squared_error
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

# Suppress warnings for cleaner output
warnings.filterwarnings('ignore')

# ---------------------------------------------------------
# Configuration & Global Variables
# ---------------------------------------------------------
FILE_PATH = FILE_PATH
TARGET_COL = 'dlq_2yrs'
PROPORTIONS = [0.1, 0.2, 0.3, 0.4, 0.5]
RANDOM_STATE = 42

# ---------------------------------------------------------
# Helper Functions
# ---------------------------------------------------------
def load_ground_truth(filepath, target_col):
    """Loads dataset and drops any existing NaNs to establish Ground Truth."""
    print("Loading dataset and establishing Ground Truth...")
    df = pd.read_csv(filepath)
    df_clean = df.dropna().reset_index(drop=True)
    print(f"Ground Truth established with {len(df_clean)} complete records.\n")
    return df_clean

def inject_mcar(df, proportion, target_col):
    """Injects Missing Completely at Random (MCAR) data into the DataFrame."""
    df_missing = df.copy()
    features = [col for col in df.columns if col != target_col]
    mask = pd.DataFrame(False, index=df.index, columns=df.columns)
    
    for col in features:
        n_missing = int(len(df) * proportion)
        missing_indices = np.random.choice(df.index, n_missing, replace=False)
        df_missing.loc[missing_indices, col] = np.nan
        mask.loc[missing_indices, col] = True
        
    return df_missing, mask

def apply_imputation(df_missing, method, **params):
    """Applies the 5 selected imputation algorithms with dynamic hyperparameter support."""
    df_imputed = df_missing.copy()
    features = [col for col in df_imputed.columns if col != TARGET_COL]
    
    if method == 'Mean':
        imputer = SimpleImputer(strategy='mean')
        df_imputed[features] = imputer.fit_transform(df_imputed[features])
        
    elif method == 'KNN':
        k = params.get('n_neighbors', 5)
        scaler = StandardScaler()
        scaled_features = scaler.fit_transform(df_imputed[features])
        imputer = KNNImputer(n_neighbors=k)
        imputed_scaled = imputer.fit_transform(scaled_features)
        df_imputed[features] = scaler.inverse_transform(imputed_scaled)
        
    elif method == 'SHD':
        df_imputed[features] = df_imputed[features].ffill().bfill()
        
    elif method == 'MissForest':
        n_est = params.get('n_estimators', 20)
        imputer = IterativeImputer(
            estimator=RandomForestRegressor(n_estimators=n_est, random_state=RANDOM_STATE, n_jobs=-1),
            max_iter=5, random_state=RANDOM_STATE
        )
        df_imputed[features] = imputer.fit_transform(df_imputed[features])
        
    elif method == 'MICE':
        max_iter = params.get('max_iter', 10)
        scaler = StandardScaler()
        scaled_features = scaler.fit_transform(df_imputed[features])
        imputer = IterativeImputer(
            estimator=BayesianRidge(),
            max_iter=max_iter, random_state=RANDOM_STATE
        )
        imputed_scaled = imputer.fit_transform(scaled_features)
        df_imputed[features] = scaler.inverse_transform(imputed_scaled)
        
    return df_imputed

def calculate_nrmse(df_true, df_imputed, mask):
    """Calculates Normalized Root Mean Squared Error specifically on missing cells."""
    features = [col for col in df_true.columns if col != TARGET_COL]
    nrmse_list = []
    
    for col in features:
        missing_idx = mask[mask[col] == True].index
        if len(missing_idx) == 0: continue
            
        y_true = df_true.loc[missing_idx, col]
        y_pred = df_imputed.loc[missing_idx, col]
        
        rmse = np.sqrt(mean_squared_error(y_true, y_pred))
        std_val = df_true[col].std()
        nrmse = rmse / std_val if std_val > 0 else 0
        nrmse_list.append(nrmse)
        
    return np.mean(nrmse_list)

def calculate_kendall_w(pivot_df, ascending=True):
    """Calculates Kendall's W (Coefficient of Concordance) for ranking consistency."""
    ranks = pivot_df.rank(axis=1, ascending=ascending)
    m, n = ranks.shape 
    
    if m < 2 or n < 2:
        return np.nan, np.nan
        
    R_i = ranks.sum(axis=0)
    R_mean = m * (n + 1) / 2
    S = np.sum((R_i - R_mean) ** 2)
    W = (12 * S) / (m**2 * (n**3 - n))
    
    chi2_stat = m * (n - 1) * W
    p_value = ss.chi2.sf(chi2_stat, n - 1)
    
    return W, p_value

# ---------------------------------------------------------
# Workflow Approach 1: Imputation Quality (With Tuning)
# ---------------------------------------------------------
def run_approach_1(df_ground_truth):
    print("="*60)
    print("APPROACH 1: IMPUTATION QUALITY (Hyperparameter Tuning)")
    print("="*60)
    
    imputation_methods = ['Mean', 'KNN', 'SHD', 'MissForest', 'MICE']
    
    # Define hyperparameter grid for tunable algorithms
    hyperparameters = {
        'Mean': [{}],
        'SHD': [{}],
        'KNN': [{'n_neighbors': 5}, {'n_neighbors': 10}, {'n_neighbors': 15}, {'n_neighbors': 20}, {'n_neighbors': 25}, {'n_neighbors': 30}],
        'MissForest': [{'n_estimators': 10}, {'n_estimators': 20}, {'n_estimators': 30}, {'n_estimators': 40}, {'n_estimators': 50}],
        'MICE': [{'max_iter': 5}, {'max_iter': 10}, {'max_iter': 15}, {'max_iter': 20}]
    }
    
    results = []
    best_params_dict = {}
    
    for prop in PROPORTIONS:
        print(f"\nInjecting {int(prop*100)}% MCAR missing data...")
        df_missing, mask = inject_mcar(df_ground_truth, prop, TARGET_COL)
        
        for method in imputation_methods:
            best_nrmse = float('inf')
            best_param = {}
            best_label = method
            
            # Grid Search over hyperparameters
            for param in hyperparameters[method]:
                df_imputed = apply_imputation(df_missing, method, **param)
                nrmse = calculate_nrmse(df_ground_truth, df_imputed, mask)
                
                # Select the parameter combination with the lowest NRMSE
                if nrmse < best_nrmse:
                    best_nrmse = nrmse
                    best_param = param
                    if param:
                        param_str = ", ".join([f"{k}={v}" for k, v in param.items()])
                        best_label = f"{method} ({param_str})"
                    else:
                        best_label = method
            
            # Save the best parameters to pass onto Approach 2
            best_params_dict[(prop, method)] = best_param
            
            results.append({
                'Missing Proportion (%)': int(prop*100),
                'Imputation Method': method,  # Keep base name for pivot table indexing
                'Tuned Label': best_label,    # Store the formatted tuning label
                'NRMSE': best_nrmse
            })
            print(f"  [{best_label}] Best NRMSE: {best_nrmse:.4f}")
            
    return pd.DataFrame(results), best_params_dict

# ---------------------------------------------------------
# Workflow Approach 2: Predictive Integrity
# ---------------------------------------------------------
def run_approach_2(df_ground_truth, best_params_dict):
    print("\n" + "="*60)
    print("APPROACH 2: PREDICTIVE INTEGRITY (Using Tuned Parameters)")
    print("="*60)
    
    imputation_methods = ['Mean', 'KNN', 'SHD', 'MissForest', 'MICE']
    classifiers = {
        'Logistic Regression': LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
        'Random Forest': RandomForestClassifier(n_estimators=50, random_state=RANDOM_STATE, n_jobs=-1),
        'XGBoost': XGBClassifier(eval_metric='logloss', random_state=RANDOM_STATE)
    }
    
    results = []
    train_df, test_df = train_test_split(df_ground_truth, test_size=0.3, random_state=RANDOM_STATE, stratify=df_ground_truth[TARGET_COL])
    X_test, y_test = test_df.drop(columns=[TARGET_COL]), test_df[TARGET_COL]
    scaler = StandardScaler()
    
    for prop in PROPORTIONS:
        print(f"\nInjecting {int(prop*100)}% MCAR missing data into Train Set...")
        train_missing, _ = inject_mcar(train_df, prop, TARGET_COL)
        y_train = train_missing[TARGET_COL]
        
        for method in imputation_methods:
            # Apply the best hyperparameter found from Approach 1 for this specific proportion
            tuned_params = best_params_dict.get((prop, method), {})
            train_imputed = apply_imputation(train_missing, method, **tuned_params)
            X_train_imputed = train_imputed.drop(columns=[TARGET_COL])
            
            for clf_name, clf in classifiers.items():
                if clf_name == 'Logistic Regression':
                    X_train_scaled = scaler.fit_transform(X_train_imputed)
                    X_test_scaled = scaler.transform(X_test)
                    clf.fit(X_train_scaled, y_train)
                    y_pred_proba = clf.predict_proba(X_test_scaled)[:, 1]
                else:
                    clf.fit(X_train_imputed, y_train)
                    y_pred_proba = clf.predict_proba(X_test)[:, 1]
                
                auc = roc_auc_score(y_test, y_pred_proba)
                results.append({
                    'Missing Proportion (%)': int(prop*100),
                    'Imputation Method': method,
                    'Classifier': clf_name,
                    'AUC': auc
                })
            print(f"  [{method}] Models trained using tuned parameters.")

    return pd.DataFrame(results)

# ---------------------------------------------------------
# Visualization Functions
# ---------------------------------------------------------
def plot_approach_1(df_results):
    plt.figure(figsize=(10, 6))
    sns.lineplot(data=df_results, x='Missing Proportion (%)', y='NRMSE', hue='Imputation Method', marker='o', linewidth=2)
    plt.title('Approach 1: Imputation Precision (NRMSE) vs. Missing Data')
    plt.xlabel('Proportion of Missing Data (%)')
    plt.ylabel('NRMSE (Lower is Better)')
    plt.grid(True)
    plt.tight_layout()
    plt.savefig('approach1_nrmse_plot.png')
    plt.close()

def plot_approach_2(df_results):
    classifiers = df_results['Classifier'].unique()
    fig, axes = plt.subplots(1, len(classifiers), figsize=(18, 5), sharey=False)
    
    for i, clf in enumerate(classifiers):
        sns.lineplot(data=df_results[df_results['Classifier'] == clf], 
                     x='Missing Proportion (%)', y='AUC', hue='Imputation Method', 
                     marker='o', ax=axes[i])
        axes[i].set_title(f'Predictive Accuracy in {clf}')
        axes[i].set_xlabel('Proportion of Missing Data (%)')
        axes[i].set_ylabel('AUC')
        axes[i].grid(True)
        
    plt.tight_layout()
    plt.savefig('approach2_classifiers_auc.png')
    plt.close()

# ---------------------------------------------------------
# Main Execution Block
# ---------------------------------------------------------
if __name__ == "__main__":
    try:
        df_ground_truth = load_ground_truth(FILE_PATH, TARGET_COL)
        
        # --- Run Experiments ---
        results_approach_1, best_params_dict = run_approach_1(df_ground_truth)
        results_approach_2 = run_approach_2(df_ground_truth, best_params_dict)
        
        # --- Visualizations ---
        print("\nGenerating visualizations...")
        plot_approach_1(results_approach_1)
        plot_approach_2(results_approach_2)
        
        # --- Statistical Analysis & Output ---
        print("\n" + "="*80)
        print("STATISTICAL ANALYSIS & RESULTS")
        print("="*80)
        
        # ---------------------------------------------------------
        # Approach 1 Output
        # ---------------------------------------------------------
        # Pivot table for the Labeled Tuning Values
        label_pivot = results_approach_1.pivot(index='Imputation Method', columns='Missing Proportion (%)', values='Tuned Label')
        print("\n[Approach 1] Best Hyperparameters Chosen per Proportion:")
        print("-" * 80)
        print(label_pivot)
        
        # Numeric Pivot table for NRMSE calculations
        pivot_1 = results_approach_1.pivot(index='Imputation Method', columns='Missing Proportion (%)', values='NRMSE')
        print("\n[Approach 1] NRMSE of imputation methods (Lower is Better):")
        print("-" * 80)
        print(pivot_1.round(4))
        
        # Ranking Table
        rank_1 = pivot_1.rank(axis=0, ascending=True).astype(int)
        rank_1['Rank of method by mean'] = rank_1.mean(axis=1).round(1)
        print("\n[Approach 1] Rank of imputation methods by NRMSE:")
        print("-" * 80)
        print(rank_1)
        
        W_1, p_1 = calculate_kendall_w(pivot_1.T, ascending=True)
        print(f"\nKendall's Statistics: W = {W_1:.3f}, p-value = {p_1:.5f}")
        
        # ---------------------------------------------------------
        # Approach 2 Output
        # ---------------------------------------------------------
        print("\n" + "-"*80)
        print("[Approach 2] AUC by Classifier & Kendall's W Consistency Tests:")
        print("-" * 80)
        classifiers = results_approach_2['Classifier'].unique()
        
        for clf in classifiers:
            clf_df = results_approach_2[results_approach_2['Classifier'] == clf]
            pivot_clf = clf_df.pivot(index='Imputation Method', columns='Missing Proportion (%)', values='AUC')
            W_clf, p_clf = calculate_kendall_w(pivot_clf.T, ascending=False)
            
            print(f"\n-> {clf} AUC Pivot Table (Higher is Better):")
            print(pivot_clf.round(4))
            
            rank_clf = pivot_clf.rank(axis=0, ascending=False).astype(int)
            rank_clf['Rank of method by mean'] = rank_clf.mean(axis=1).round(1)
            print(f"\n-> {clf} Rank of imputation methods by AUC:")
            print(rank_clf)
            print(f"\n   Kendall's W (Consistency of ranks in {clf}) = {W_clf:.3f}, p-value = {p_clf:.5f}")
            print("-" * 80)
            
    except FileNotFoundError:
        print(f"Error: Could not find the dataset at '{FILE_PATH}'.")

Loading dataset and establishing Ground Truth...
Ground Truth established with 16714 complete records.

APPROACH 1: IMPUTATION QUALITY (Hyperparameter Tuning)

Injecting 10% MCAR missing data...
  [Mean] Best NRMSE: 0.9393
  [KNN (n_neighbors=30)] Best NRMSE: 0.6701
  [SHD] Best NRMSE: 1.6732
  [MissForest (n_estimators=50)] Best NRMSE: 0.5785
  [MICE (max_iter=5)] Best NRMSE: 0.6266

Injecting 20% MCAR missing data...
  [Mean] Best NRMSE: 0.9332
  [KNN (n_neighbors=30)] Best NRMSE: 0.7973
  [SHD] Best NRMSE: 1.3700
  [MissForest (n_estimators=50)] Best NRMSE: 0.6677
  [MICE (max_iter=5)] Best NRMSE: 0.6396

Injecting 30% MCAR missing data...
  [Mean] Best NRMSE: 1.0163
  [KNN (n_neighbors=15)] Best NRMSE: 0.9324
  [SHD] Best NRMSE: 1.3508
  [MissForest (n_estimators=50)] Best NRMSE: 0.7631
  [MICE (max_iter=5)] Best NRMSE: 0.8503

Injecting 40% MCAR missing data...
  [Mean] Best NRMSE: 0.9604
  [KNN (n_neighbors=30)] Best NRMSE: 0.9220
  [SHD] Best NRMSE: 1.3546
  [MissForest (n_estim